In [1]:
import os
from pathlib import Path
import json
from datetime import datetime
import asyncio
import nest_asyncio
from concurrent.futures import ThreadPoolExecutor
from deepeval import evaluate
from deepeval.evaluate import CacheConfig, AsyncConfig
from deepeval.metrics import GEval, AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.dataset import EvaluationDataset
import logfire
from tqdm.notebook import tqdm
from dotenv import load_dotenv
load_dotenv()
logfire.configure(token=os.environ.get("LOGFIRE_TOKEN"), console=False)
logfire.instrument_openai()
from agent import MyAgent

Overriding of current TracerProvider is not allowed


In [2]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
timestamp

'20250723_1409'

In [3]:
nest_asyncio.apply()
executor = ThreadPoolExecutor(max_workers=3)
semaphore = asyncio.Semaphore(3)

In [4]:
# QS_PATH = Path("generated_QUERIES_Polish_16_07.json")
QS_PATH=Path('./synth_datasets/raw_queries/Polish_20250721_1305_10_gpt-4o.json')

In [5]:
with open(QS_PATH, "r", encoding="utf-8") as f:
    qs_ans = json.load(f)

In [6]:
def get_retrival_context(input: str) -> str:
    q = [q for q in qs_ans if q["question"]==input][0]
    filename = q['filename']

    with open(f"./data/events/{filename}",'r', encoding='utf-8') as f:
        content = f.read()

    return content

In [7]:
retrival_context_dict = {q["question"]:get_retrival_context(q["question"]) for q in qs_ans}

In [ ]:
async def generate_test_case(question: str, expected_answer: str) -> list[LLMTestCase]:
    async with semaphore:
        loop = asyncio.get_event_loop()
        answer = await loop.run_in_executor(
            executor,
            MyAgent(read_n_eval_model="gpt-4.1").step,
            question
        )
        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            expected_output=expected_answer,
            retrieval_context=[retrival_context_dict[question]]
        )
        return test_case

async def generate_dataset(qs_ans: list[dict]) -> list[LLMTestCase]:
    tasks = []
    for q in qs_ans:
        question = q["question"]
        expected_answer = q["answer"]
        tasks.append(generate_test_case(question, expected_answer))
    all_test_cases = []
    with tqdm(total=len(tasks), desc="Generating test cases") as pbar:
        for coro in asyncio.as_completed(tasks):
            try:
                result = await coro
                all_test_cases.append(result)
                pbar.update(1)
            except Exception as e:
                print(f"Error processing task: {e}")
                pbar.update(1)
                continue
    return all_test_cases

@logfire.instrument("Generate Eval Dataset", extract_args=True, record_return=True)
async def main():
    qs = qs_ans[:15]
    all_test_cases = await generate_dataset(qs)
    return all_test_cases

In [9]:
test_cases = await main()

Generating test cases:   0%|          | 0/10 [00:00<?, ?it/s]

Successfully connected to existing collection 'event_semantic_summaries' with 424 documents.
Successfully connected to existing collection 'event_semantic_summaries' with 424 documents.
Successfully connected to existing collection 'event_semantic_summaries' with 424 documents.

----- Step 1 -----


----- Step 1 -----


----- Step 1 -----

Action: parse_user_query

Think: Użytkownik szuka konkursów o tematyce marynistycznej, które odbywają się w Warszawie pod koniec czerwca. Najpierw wyodrębnię te informacje z zapytania, aby móc skutecznie wyszukać odpowiednie wydarzenia.

Result: Think: Użytkownik szuka konkursów o tematyce marynistycznej w Warszawie pod koniec czerwca. "Pod koniec czerwca" oznacza zakres dat od około 21 do 30 czerwca 2025. Miasto jest jasno określone jako Warszawa. Kluczowe słowa to "konkursy" i "marynistyczne". Zapytanie zostanie przekształcone w pełne zdanie opisujące wydarzenia konkursowe o tematyce marynistycznej w Warszawie pod koniec czerwca 2025.
Parsed user q

In [10]:
dataset = EvaluationDataset()
# dataset.add_test_cases_from_json_file(
#     file_path="./deval_datasets/queries_polish_16_07_20250717_2206.json",
#     input_key_name="input",
#     actual_output_key_name="actual_output",
#     expected_output_key_name="expected_output",
#     retrieval_context_key_name="retrieval_context"

# )

In [12]:
for test_case in test_cases:
    dataset.add_test_case(test_case)

In [13]:
dataset.save_as(file_type="json", directory="./synth_datasets/deval_datasets/", include_test_cases=True, file_name=f"Polish_20250721_1305_10_gpt-4o{timestamp}")

Evaluation dataset saved at ./synth_datasets/deval_datasets/Polish_20250721_1305_10_gpt-4o20250723_1409.json!


'./synth_datasets/deval_datasets/Polish_20250721_1305_10_gpt-4o20250723_1409.json'

In [ ]:
custom_metric = GEval(
    name="Golden Answer Correctness",
    model="gpt-4.1-mini",
    criteria="""You are evaluating an AI event search agent. The 'actual_output' is a list of events the agent found. The 'expected_output' is the single, specific event that the agent was *required* to find and extract correctly.

    Your evaluation must follow a two-step process:
    1.  **Retrieval Check:** First, determine if the event from the 'expected_output' is present anywhere in the 'actual_output' list. You should match based on the event title. If the expected event is NOT found in the list, the score is 0.
    2.  **Extraction Check:** If the event was found, you must then compare the date and location details for that specific event in the 'actual_output' against the date and location in the 'expected_output'. The score should reflect how accurately these details were extracted. Minor formatting differences are acceptable, but incorrect dates or locations are major failures.

    Score from 0.0 (the required event was not found at all) to 1.0 (the event was found and all its key details were extracted correctly).""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    threshold=0.7 # A reasonably high bar for success
)

In [14]:
custom_metric = GEval( ##TODO check the new definiotnin and push back, suggest using actions instead of criteria 
    name="Factual Correctness",
    model="gpt-4.1-mini",
    # criteria="Determine whether the 'actual_output' event details (location, title, date) contain the information present in the 'expected_output' event details. Slight deviations in location name are acceptable.",
    evaluation_steps=[
        "Step 1: Scan the 'actual_output', which may contain a list of several events, and determine if any event in the list has a title that EXACTLY or VERY CLOSELY matches the title in the 'expected_output'.",
        "Step 2: Verify that the date(s) of that specific event in the 'actual_output' align with the date(s) in the 'expected_output'. Minor formatting differences are acceptable as long as the core date is correct.",
        "Step 3: Verify that the location in the 'actual_output' aligns with the location in the 'expected_output'. Minor variations (e.g. missing postal code, or truncated location names) are acceptable."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    threshold=0.5)

answer_relevancy_metric = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4.1-mini",
    include_reason=True
)

faithfulness_metric = FaithfulnessMetric(
    threshold=0.7,
    model="gpt-4.1-mini",
    include_reason=True
)

In [15]:
with logfire.span("Evaluating Dataset"):
    evaluate(
        test_cases=dataset.test_cases,
        metrics=[custom_metric,
                answer_relevancy_metric,
                faithfulness_metric],
        cache_config=CacheConfig(use_cache=True),
        async_config=AsyncConfig(
            run_async=True,
            max_concurrent=5,
            throttle_value=2))

✨ You're running DeepEval's latest Factual Correctness [GEval] Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Factual Correctness [GEval] (score: 0.8979667646249219, threshold: 0.5, strict: False, evaluation model: gpt-4.1-mini, reason: The actual output's event title closely matches the expected title with only minor differences in quotation marks and capitalization. The start and end dates align correctly despite different formatting, and the location matches well with minor capitalization differences. The only slight shortcoming is the title's minor variation in quotation marks and the location's capitalization inconsistency., error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4.1-mini, reason: The score is 1.00 because the response is fully relevant to the input about the opening of a new gallery at the Polin Museum in late June 2025, with no irrelevant statements present., error: None)
  - ✅ Faithfulness (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4.1-mini, reason: The score is 1.00 because t

✓ Tests finished 🎉! Run 'deepeval view' to analyze, debug, and save evaluation results on Confident AI.

In [ ]:
def get_question(input: str) -> None:
    q = [q for q in qs_ans if q["question"]==input][0]
    print(f"Question: {q['question']}")
    print(f"Answer: {q['answer']}")
    print(f"filename: {q['filename']}")

get_question("Warsztaty japońskiego malarstwa tuszem w Warszawie w lipcu 2025")

In [16]:
OUTPUT_DIR=Path(f"./.deepeval/run_{timestamp}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
os.rename("./.deepeval/.deepeval-cache.json", OUTPUT_DIR/f".deepeval-cache_{timestamp}.json")
os.rename("./.deepeval/.deepeval_telemetry.txt", OUTPUT_DIR/f".deepeval_telemetry_{timestamp}.txt")
os.rename("./.deepeval/.latest_test_run.json", OUTPUT_DIR/f".latest_test_run{timestamp}.json")